In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
from torch import float32, logical_and, logical_or, logical_xor, randn
from torch.nn import Module
from torch.optim import SGD

import import_ipynb
import sigmoid # type: ignore
import linear # type: ignore
import binary_cross_entropy # type: ignore
from common import average_run, T # type: ignore

In [ ]:
class PerceptronLog(Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        #
        # For demonstration purposes
        #

        self.linear =  linear.Linear(in_features, out_features)
        self.sigmoid = sigmoid.Sigmoid()

        #
        # [Linear Layer] + [BinaryCrossEntropyWithLogits = Sigmoid + BinaryCrossEntropy] 
        # is more numerically stable than [Linear] + [Sigmoid] + [BinaryCrossEntropy]
        #

    def forward(self, x):
        self.z = self.linear(x)
        self.p = self.sigmoid(self.z)
        return self.p
    
    # Extension
    # To facilitate testing and experimentation various perceptrons.
    def get_loss_fn(self):
        return binary_cross_entropy.BinaryCrossEntropy()
    
    # Extension
    # To facilitate testing and experimentation various perceptrons.
    def predict(self, x):
        return self.forward(x) > 0.5
    

def test_perceptron(perceptron_type, X, Y, epochs):
    model = perceptron_type(X.shape[1], 1)
    optimizer = SGD(model.parameters(), lr=0.1)

    for _ in range(epochs):
        optimizer.zero_grad()

        P = model(X)
        loss = model.get_loss_fn()(P, Y)
        
        loss.backward()
        optimizer.step()

    return (model.predict(X) == Y).mean(dtype=float32).item()


def test_perceptron_boolean(perceptron_type, bool_fn, epochs, samples=100):
    X = (randn(samples, 2, dtype=float32) > 0).float()
    Y = bool_fn(X[:, 0], X[:, 1]).float().unsqueeze(1)
    return test_perceptron(perceptron_type, X, Y, epochs)


if __name__ == "__main__":
    logical_nand = lambda x1, x2: logical_and(x1, x2).logical_not()

    print(f" AND, 100 epochs: {average_run(10, lambda: test_perceptron_boolean(PerceptronLog, logical_and, epochs=100)):.2f}")
    print(f" AND, 500 epochs: {average_run(10, lambda: test_perceptron_boolean(PerceptronLog, logical_and, epochs=500)):.2f}")
    print(f"  OR, 100 epochs: {average_run(10, lambda: test_perceptron_boolean(PerceptronLog, logical_or, epochs=100)):.2f}")
    print(f"  OR, 500 epochs: {average_run(10, lambda: test_perceptron_boolean(PerceptronLog, logical_or, epochs=500)):.2f}")
    print(f"NAND, 100 epochs: {average_run(10, lambda: test_perceptron_boolean(PerceptronLog, logical_nand, epochs=100)):.2f}")
    print(f"NAND, 500 epochs: {average_run(10, lambda: test_perceptron_boolean(PerceptronLog, logical_nand, epochs=500)):.2f}")

    # XOR is not linearly separable, so the accuracy should be around 0.5 (random guessing).
    print(f" XOR, 100 epochs: {average_run(10, lambda: test_perceptron_boolean(PerceptronLog, logical_xor, epochs=100)):.2f}")
    print(f" XOR, 500 epochs: {average_run(10, lambda: test_perceptron_boolean(PerceptronLog, logical_xor, epochs=500)):.2f}")

 AND, 100 epochs: 0.81
 AND, 500 epochs: 1.00
  OR, 100 epochs: 0.78
  OR, 500 epochs: 1.00
NAND, 100 epochs: 0.76
NAND, 500 epochs: 1.00
 XOR, 100 epochs: 0.59
 XOR, 500 epochs: 0.61
